# Full Hexagon Segmentation Pipeline

Reads `02_merged_data.parquet`, assigns H3 hexagons at a configurable resolution, and visualises trip-count and feature distributions per hexagon.

**Configuration** — change the variables in the cell below:
- `H3_RESOLUTION` — hexagon size (0 = coarsest, 15 = finest; 7≈5 km, 8≈1 km, 9≈0.3 km)
- `FEATURE` — numeric column whose mean per hexagon is plotted


In [ ]:
import pandas as pd
import polars as pl
import h3
import matplotlib.pyplot as plt
import numpy as np
import folium
import branca.colormap as cm
import ipywidgets as widgets
from IPython.display import display


## Configuration


In [49]:
H3_RESOLUTION = 8    # 0-15; higher = smaller hexagons
FEATURE       = "fare_usd"  # any numeric column (e.g. trip_miles, tips_usd, temperature_2m)
TOP_N         = 20   # hexagons shown in bar charts


## Load data


In [50]:
df = pl.read_parquet("../data/02_merged_data.parquet").to_pandas()
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nNumeric columns available for FEATURE:")
print(sorted(df.select_dtypes(include='number').columns.tolist()))
display(df.head(3))


Shape: 478,114 rows x 34 columns

Numeric columns available for FEATURE:
['apparent_temperature', 'different_days', 'dropoff_census_tract', 'dropoff_community_area', 'dropoff_lat', 'dropoff_lon', 'extras_usd', 'fare_usd', 'pickup_census_tract', 'pickup_community_area', 'pickup_lat', 'pickup_lon', 'precipitation', 'rain', 'relative_humidity_2m', 'snow_depth', 'snowfall', 'temperature_2m', 'temperature_2m_max', 'temperature_2m_min', 'tips_usd', 'tolls_usd', 'trip_miles', 'trip_seconds', 'trip_total_usd', 'wind_direction_10m', 'wind_gusts_10m', 'wind_speed_10m']


,trip_id,taxi_id,trip_start,trip_end,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,precipitation,wind_speed_10m,wind_direction_10m,wind_gusts_10m,apparent_temperature,relative_humidity_2m,snow_depth,snowfall,temperature_2m_max,temperature_2m_min
0,fff4a20bb745a6cc7668c2c5cc13f24d0396995a,bd2bb222f12bbd1e738432cafdf7d8c345799df3118510...,2026-03-16 16:45:00-05:00,2026-03-16 17:00:00,745.0,1.94,1.703128e+10,1.703108e+10,28.0,8.0,...,0.1,26.459387,303.906311,63.360001,-10.429605,71.838501,0.03,0.07,0.95,-6.3
1,feebcd1d11c554c3675e93ad4e747c11ebcc07f9,db3d7dbdc135e7fe12b5b92e1eee281d85025028652bc7...,2026-03-16 16:45:00-05:00,2026-03-16 17:00:00,843.0,2.33,NaN,NaN,24.0,28.0,...,0.1,26.459387,303.906311,63.360001,-10.429605,71.838501,0.03,0.07,0.95,-6.3
2,feb84286650933078d1b98dde1d9b140485eed83,17a313ad40ea410818634782c7202abda7ce5289dc03f9...,2026-03-16 16:45:00-05:00,2026-03-16 17:00:00,969.0,2.67,NaN,NaN,32.0,7.0,...,0.1,26.459387,303.906311,63.360001,-10.429605,71.838501,0.03,0.07,0.95,-6.3


## Assign H3 hexagons


In [51]:
def to_h3_cell(lat, lon, resolution):
    if pd.isna(lat) or pd.isna(lon):
        return None
    return h3.latlng_to_cell(float(lat), float(lon), resolution)

df["start_h3"] = [to_h3_cell(lat, lon, H3_RESOLUTION) for lat, lon in zip(df["pickup_lat"],  df["pickup_lon"])]
df["end_h3"]   = [to_h3_cell(lat, lon, H3_RESOLUTION) for lat, lon in zip(df["dropoff_lat"], df["dropoff_lon"])]

print(f"H3 resolution           : {H3_RESOLUTION}")
print(f"Unique pickup hexagons  : {df['start_h3'].nunique():,}")
print(f"Unique dropoff hexagons : {df['end_h3'].nunique():,}")
print(f"Null pickup h3          : {df['start_h3'].isna().sum():,}")
print(f"Null dropoff h3         : {df['end_h3'].isna().sum():,}")


H3 resolution           : 8
Unique pickup hexagons  : 179
Unique dropoff hexagons : 208
Null pickup h3          : 858
Null dropoff h3         : 8,920


In [52]:
start_counts = df["start_h3"].dropna().value_counts().reset_index()
start_counts.columns = ["h3_cell", "trip_count"]
end_counts = df["end_h3"].dropna().value_counts().reset_index()
end_counts.columns = ["h3_cell", "trip_count"]

## Map: trip count per hexagon

Toggle between **Pickup hexagons** and **Dropoff hexagons** using the layer control (top-right). Colour intensity encodes number of trips; hover a hexagon for the exact count.

In [53]:
all_lats = pd.concat([df["pickup_lat"], df["dropoff_lat"]], ignore_index=True).dropna()
all_lons = pd.concat([df["pickup_lon"], df["dropoff_lon"]], ignore_index=True).dropna()
map_center = [all_lats.mean(), all_lons.mean()]

max_count = int(max(
    start_counts["trip_count"].max() if not start_counts.empty else 0,
    end_counts["trip_count"].max()   if not end_counts.empty   else 0,
))
count_colormap = cm.linear.YlOrRd_09.scale(0, max_count)
count_colormap.caption = "Trips per hexagon"

m_trips = folium.Map(location=map_center, zoom_start=11, tiles="CartoDB positron", width=1200, height=600)


def _add_hex_layer(map_obj, counts_df, layer_name, show, colormap):
    layer = folium.FeatureGroup(name=layer_name, show=show)
    for _, row in counts_df.iterrows():
        boundary = h3.cell_to_boundary(row["h3_cell"])
        color = colormap(row["trip_count"])
        folium.Polygon(
            locations=[[lat, lon] for lat, lon in boundary],
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.65,
            tooltip=folium.Tooltip(f"{layer_name}: {row['trip_count']:,} trips"),
        ).add_to(layer)
    layer.add_to(map_obj)


_add_hex_layer(m_trips, start_counts, "Pickup hexagons",  True,  count_colormap)
_add_hex_layer(m_trips, end_counts,   "Dropoff hexagons", False, count_colormap)
count_colormap.add_to(m_trips)
folium.LayerControl(collapsed=False).add_to(m_trips)

display(m_trips)


## Map: feature value per hexagon (interactive)

Use the **dropdown** to pick any numeric feature. The map re-renders immediately, colouring each pickup hexagon by its **mean value**. Hover a hexagon to see the exact value and trip count.

In [ ]:
SELECTABLE_FEATURES = [
    "apparent_temperature", "different_days", "dropoff_census_tract",
    "dropoff_community_area", "dropoff_lat", "dropoff_lon", "extras_usd",
    "fare_usd", "pickup_census_tract", "pickup_community_area", "pickup_lat",
    "pickup_lon", "precipitation", "rain", "relative_humidity_2m", "snow_depth",
    "snowfall", "temperature_2m", "temperature_2m_max", "temperature_2m_min",
    "tips_usd", "tolls_usd", "trip_miles", "trip_seconds", "trip_total_usd",
    "wind_direction_10m", "wind_gusts_10m", "wind_speed_10m",
]
# Keep only features that actually exist in the loaded dataframe
SELECTABLE_FEATURES = [f for f in SELECTABLE_FEATURES if f in df.columns]

_all_lats = pd.concat([df["pickup_lat"], df["dropoff_lat"]], ignore_index=True).dropna()
_all_lons = pd.concat([df["pickup_lon"], df["dropoff_lon"]], ignore_index=True).dropna()
_map_center = [_all_lats.mean(), _all_lons.mean()]


def _build_feature_map(feature):
    agg = (
        df.dropna(subset=["start_h3", feature])
        .groupby("start_h3")[feature]
        .agg(["mean", "count"])
        .reset_index()
    )
    agg.columns = ["h3_cell", "mean_val", "trip_count"]

    feat_min, feat_max = agg["mean_val"].min(), agg["mean_val"].max()
    colormap = cm.linear.PuBuGn_09.scale(feat_min, feat_max)
    colormap.caption = f"Mean {feature} per hexagon"

    m = folium.Map(location=_map_center, zoom_start=11, tiles="CartoDB positron", width=1200, height=600)
    layer = folium.FeatureGroup(name=f"Mean {feature}", show=True)

    for _, row in agg.iterrows():
        boundary = h3.cell_to_boundary(row["h3_cell"])
        color = colormap(row["mean_val"])
        folium.Polygon(
            locations=[[lat, lon] for lat, lon in boundary],
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            tooltip=folium.Tooltip(
                f"Mean {feature}: {row['mean_val']:.2f}<br>Trips: {row['trip_count']:,}"
            ),
        ).add_to(layer)

    layer.add_to(m)
    colormap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m


_initial = FEATURE if FEATURE in SELECTABLE_FEATURES else SELECTABLE_FEATURES[0]

_dropdown = widgets.Dropdown(
    options=SELECTABLE_FEATURES,
    value=_initial,
    description="Feature:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
_out = widgets.Output()


def _on_change(change):
    with _out:
        _out.clear_output(wait=True)
        display(_build_feature_map(change["new"]))


_dropdown.observe(_on_change, names="value")

with _out:
    display(_build_feature_map(_initial))

display(widgets.VBox([_dropdown, _out]))


## Map: trips per hexagon by day of week

Select a **day of the week** (or "All days") and toggle between **Pickup** and **Dropoff** hexagons. Colour intensity encodes number of trips for that day. `different_days` encodes 0 = Monday … 6 = Sunday.

In [57]:
_DOW_NAMES = {
    0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday",
    4: "Friday", 5: "Saturday", 6: "Sunday",
}

_dow_day_dropdown = widgets.Dropdown(
    options=[("All days", -1)] + [(name, i) for i, name in _DOW_NAMES.items()],
    value=-1,
    description="Day:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_dow_hex_dropdown = widgets.Dropdown(
    options=[("Pickup hexagons", "start_h3"), ("Dropoff hexagons", "end_h3")],
    value="start_h3",
    description="Trip end:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_dow_out = widgets.Output()


def _build_dow_map(day_val, h3_col):
    subset = df if day_val == -1 else df[df["different_days"] == day_val]

    counts = (
        subset.dropna(subset=[h3_col])[h3_col]
        .value_counts()
        .reset_index()
    )
    counts.columns = ["h3_cell", "trip_count"]

    day_label = "All days" if day_val == -1 else _DOW_NAMES[day_val]
    hex_label = "Pickup" if h3_col == "start_h3" else "Dropoff"

    m = folium.Map(location=_map_center, zoom_start=11, tiles="CartoDB positron")

    if counts.empty:
        return m

    cmap = cm.linear.YlOrRd_09.scale(0, int(counts["trip_count"].max()))
    cmap.caption = f"{hex_label} trips per hexagon — {day_label}"

    layer = folium.FeatureGroup(name=f"{hex_label} · {day_label}", show=True)
    for _, row in counts.iterrows():
        boundary = h3.cell_to_boundary(row["h3_cell"])
        color = cmap(row["trip_count"])
        folium.Polygon(
            locations=[[lat, lon] for lat, lon in boundary],
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            tooltip=folium.Tooltip(
                f"{hex_label} · {day_label}<br>Trips: {row['trip_count']:,}"
            ),
        ).add_to(layer)

    layer.add_to(m)
    cmap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m


def _on_dow_change(change):
    with _dow_out:
        _dow_out.clear_output(wait=True)
        display(_build_dow_map(_dow_day_dropdown.value, _dow_hex_dropdown.value))


_dow_day_dropdown.observe(_on_dow_change, names="value")
_dow_hex_dropdown.observe(_on_dow_change, names="value")

with _dow_out:
    display(_build_dow_map(-1, "start_h3"))

display(widgets.VBox([
    widgets.HBox([_dow_day_dropdown, _dow_hex_dropdown]),
    _dow_out,
]))

## Map: trips per hexagon by hour of day

Select an **hour** (0–23, or "All hours") and toggle between **Pickup** and **Dropoff** hexagons. Hour is derived from `trip_start`.

In [ ]:
_trip_hour = df["trip_start"].dt.hour  # computed once, reused on every map render

_hour_day_dropdown = widgets.Dropdown(
    options=[("All hours", -1)] + [(f"{h:02d}:00", h) for h in range(24)],
    value=-1,
    description="Hour:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_hour_hex_dropdown = widgets.Dropdown(
    options=[("Pickup hexagons", "start_h3"), ("Dropoff hexagons", "end_h3")],
    value="start_h3",
    description="Trip end:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_hour_out = widgets.Output()


def _build_hour_map(hour_val, h3_col):
    mask = slice(None) if hour_val == -1 else (_trip_hour == hour_val)
    subset = df if hour_val == -1 else df[mask]

    counts = (
        subset.dropna(subset=[h3_col])[h3_col]
        .value_counts()
        .reset_index()
    )
    counts.columns = ["h3_cell", "trip_count"]

    hour_label = "All hours" if hour_val == -1 else f"{hour_val:02d}:00"
    hex_label  = "Pickup" if h3_col == "start_h3" else "Dropoff"

    m = folium.Map(location=_map_center, zoom_start=11, tiles="CartoDB positron")

    if counts.empty:
        return m

    cmap = cm.linear.BuPu_09.scale(0, int(counts["trip_count"].max()))
    cmap.caption = f"{hex_label} trips per hexagon — {hour_label}"

    layer = folium.FeatureGroup(name=f"{hex_label} · {hour_label}", show=True)
    for _, row in counts.iterrows():
        boundary = h3.cell_to_boundary(row["h3_cell"])
        color = cmap(row["trip_count"])
        folium.Polygon(
            locations=[[lat, lon] for lat, lon in boundary],
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            tooltip=folium.Tooltip(
                f"{hex_label} · {hour_label}<br>Trips: {row['trip_count']:,}"
            ),
        ).add_to(layer)

    layer.add_to(m)
    cmap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m


def _on_hour_change(change):
    with _hour_out:
        _hour_out.clear_output(wait=True)
        display(_build_hour_map(_hour_day_dropdown.value, _hour_hex_dropdown.value))


_hour_day_dropdown.observe(_on_hour_change, names="value")
_hour_hex_dropdown.observe(_on_hour_change, names="value")

with _hour_out:
    display(_build_hour_map(-1, "start_h3"))

display(widgets.VBox([
    widgets.HBox([_hour_day_dropdown, _hour_hex_dropdown]),
    _hour_out,
]))